# 02a · Handcrafted feature definitions

**Purpose.** Show each of the 12 handcrafted features (baseline level, steady state response, normalized response, peak response, time to peak, rise slope, decay slope, area under the curve, 4 EMA derivative features) drawn on one example sensor curve from each dataset.

**Inputs**
- One curve from GasWindTunnel (1500,) and one from EverydayScents (210,)

**Outputs**
- Figures in `report/figures/` (one panel per feature)

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import load_baseline_level, load_processed
from src.features import handcrafted as hc

FIG = ROOT / "report/figures"
FIG.mkdir(parents=True, exist_ok=True)

## Each feature drawn on one curve
One GasWindTunnel curve (most responsive sensor) and one EverydayScents curve (voc channel).

In [ ]:
def show_features(r, level, n_base, exp_end, title, ax_curve, ax_ema):
    """Draw the 12 features on one baseline-corrected curve r (T,)."""
    T = len(r)
    p = hc.peak_index(r, n_base, exp_end)
    n_ss = max(1, (exp_end - n_base) // 5)
    ax_curve.plot(r, color="k", lw=1, label="response r")
    ax_curve.axhline(0, color="gray", lw=0.8, label=f"(1) baseline, level = {level:.3g}")
    ax_curve.axvspan(n_base, exp_end, color="C0", alpha=0.08, label="(8) AUC window = exposure")
    ax_curve.axvspan(exp_end - n_ss, exp_end, color="C1", alpha=0.25,
                     label=f"(2) steady state = {hc.steady_state(r, n_base, exp_end):.3g}")
    ax_curve.plot(p, r[p], "rv", ms=9, label=f"(4) peak = {r[p]:.3g}, (5) time to peak = {p - n_base}")
    for seg_start, seg_end, name, color in [(n_base, p + 1, "(6) rise slope", "C2"), (p, T, "(7) decay slope", "C3")]:
        seg = r[seg_start:seg_end]
        if len(seg) >= 2:
            a, b = np.polyfit(np.arange(len(seg)), seg, 1)
            ax_curve.plot(np.arange(seg_start, seg_end), a * np.arange(len(seg)) + b, color=color, lw=2,
                          label=f"{name} = {a:.3g}")
    ax_curve.set_title(title + f"   (3) normalized = {hc.normalized(r, n_base, exp_end, level):.3g}")
    ax_curve.legend(fontsize=7)
    for alpha, color in [(0.1, "C4"), (0.01, "C5")]:
        y = hc.ema_derivative(r, n_base, alpha)
        ax_ema.plot(np.arange(n_base, n_base + len(y)), y, color=color,
                    label=f"EMA of derivative, alpha {alpha}: max {y.max():.3g}, min {y.min():.3g}")
    ax_ema.set(xlabel="time point", title="(9-12) EMA features")
    ax_ema.legend(fontsize=7)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), gridspec_kw={"height_ratios": [2, 1]})
for col, dataset in enumerate(["gwt", "es"]):
    X, meta, info = load_processed(dataset)            # (N, C, T)
    levels = load_baseline_level(dataset)              # (N, C)
    s = np.abs(X).mean(axis=(0, 2)).argmax() if dataset == "gwt" else info["channels"].index("voc")
    i = int(np.abs(X[:, s]).mean(axis=1).argmax())     # recording with the strongest response on that sensor
    title = f"{dataset}: sample {i}, sensor {s}, class {info['classes'][meta.loc[i, 'y_multi']]}"
    show_features(X[i, s].astype(float), float(levels[i, s]), info["n_base"], info["exposure_end"],
                  title, axes[0, col], axes[1, col])
plt.tight_layout()
plt.savefig(FIG / "02a_feature_definitions.png", dpi=150)
plt.show()

In [ ]:
# the 12 values of the two curves drawn above, in the order used by handcrafted_features
rows = {}
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)
    levels = load_baseline_level(dataset)
    s = np.abs(X).mean(axis=(0, 2)).argmax() if dataset == "gwt" else info["channels"].index("voc")
    i = int(np.abs(X[:, s]).mean(axis=1).argmax())
    rows[dataset] = hc.curve_features(X[i, s].astype(float), float(levels[i, s]), info["n_base"], info["exposure_end"])
pd.DataFrame(rows, index=hc.FEATURE_NAMES)